In [ ]:
from data_loading import load_ecoli

ecoli = load_ecoli("E_coli_v4_Build_6_chips907probes7459.tab")
ecoli.summary()
ecoli.plot_distribution()
ecoli.to_dataframe().head(10)
ecoli.plot_heatmap()

In [ ]:
import numpy as np

X = ecoli.X
genes = ecoli.genes
samples = ecoli.samples

print(f"\nReady for MCbiclust:")
print(f"  X shape:  {X.shape}")
print(f"  genes:    {len(genes)}")
print(f"  samples:  {len(samples)}")

all_gene_indices = np.arange(len(genes))

In [ ]:
from correlation import avg_abs_corr_rows, plot_gene_corr_heatmap
alpha_global = avg_abs_corr_rows(X)

print(f"Global alpha (all genes): {alpha_global:.4f}")

In [ ]:
plot_gene_corr_heatmap(X, genes=genes, subset_size=100)

In [ ]:
from gene_seed import select_initial_seed_genes
import pandas as pd

initial_gene_set = select_initial_seed_genes(
    X,
    n_genes=1000,
    random_state=42
)

print(
    f"Selected {len(initial_gene_set)} genes "
    f"(indices {initial_gene_set.min()}-{initial_gene_set.max()})"
)

seed_df = pd.DataFrame({
    "gene_index": initial_gene_set,
    "gene_name": genes[initial_gene_set]
})

# seed_df.head(20)

In [ ]:
from find_seed_sample import find_seed_bicluster, run_find_seed_over_iterations
from seed_plots import plot_alpha_vs_iterations, plot_sample_corr_heatmap

steps = list(range(100, 1001, 100))

seed_history, alpha_history = run_find_seed_over_iterations(
    X=X,
    gene_set=initial_gene_set,
    steps=steps,
    n_samples=10,
    random_state=42,
)

best_samples = seed_history[-1]   # best seed from the last iteration step
best_alpha   = alpha_history[-1]  # corresponding alpha
# plot_alpha_vs_iterations(steps, alpha_history, title="FindSeed: Alpha Improvement (Python)")




In [ ]:
plot_sample_corr_heatmap(X, subset_size=100, seed=42, cluster=False, vmin=0, vmax=1)
# plot_sample_corr_heatmap(X, subset_size=1000, seed=42, cluster=True, vmin=0.5, vmax=1)

In [ ]:
from pruning import prune_bicluster_genes
from pruning_plots import plot_pruning_summary


print(f"Original: {len(initial_gene_set)} genes, {len(best_samples)} samples")
print(f"Original alpha: {best_alpha:.6f}")
print()

local_idx_py, pruned_gene_set, cluster_labels_py, alphas_py, group_alphas = prune_bicluster_genes(
    X,
    gene_set=initial_gene_set,
    sample_set=best_samples,
    original_alpha=best_alpha,
    n_groups=8,
    random_state=42,
)

print()
print(f"Pruned: {len(pruned_gene_set)} genes remain")
print(f"Removed: {len(initial_gene_set) - len(pruned_gene_set)} genes")

In [ ]:
from pruning_plots import (
    plot_pruning_summary,
    plot_pruned_gene_correlation_heatmap,
)

plot_pruning_summary(
    X=X,
    initial_gene_set=initial_gene_set,
    pruned_gene_set=pruned_gene_set,
    best_samples=best_samples,
    local_idx_py=local_idx_py,
    cluster_labels_py=cluster_labels_py,
    group_alphas=group_alphas,
    best_alpha=best_alpha,
    
)

In [ ]:


plot_pruned_gene_correlation_heatmap(
    X,
    pruned_gene_set,
    best_samples,
    title="Pruned Gene Set Correlation Heatmap",
    max_genes=None,
    figsize=(14, 10),
    cmap="RdBu_r"
)

In [ ]:
from sample_sort import extend_bicluster_samples_fast

ranked_samples, sample_alphas, picked = extend_bicluster_samples_fast(
    X,
    pruned_gene_set,
    best_samples,
    progress_every=50,
    print_only_improvements=False,
    max_rank=None,
)

In [ ]:
np.save("ranked_samples.npy", ranked_samples)
np.save("sample_alphas.npy", np.array(sample_alphas))
np.save("pruned_gene_set.npy", pruned_gene_set)
np.save("best_samples.npy", best_samples)
import json
with open("picked.json", "w") as f:
    json.dump([(int(s), float(a), float(d)) for s, a, d in picked], f)
print("Saved.")

In [ ]:
import numpy as np, json

ranked_samples  = np.load("ranked_samples.npy")
sample_alphas   = np.load("sample_alphas.npy").tolist()
pruned_gene_set = np.load("pruned_gene_set.npy")
best_samples    = np.load("best_samples.npy")
with open("picked.json") as f:
    picked = [tuple(x) for x in json.load(f)]
print("Loaded.")

In [ ]:
from sample_sort import plot_sample_extension_summary

plot_sample_extension_summary(
    X             = X,
    gene_set      = pruned_gene_set,
    ranked_samples = ranked_samples,
    sample_alphas  = sample_alphas,
    sample_names   = samples,
    global_alpha   = avg_abs_corr_rows(X),
    n_seed         = 10
)

for n in [10, 20, 50, 100, 150, 200]:
    top_n = ranked_samples[:n]
    a = avg_abs_corr_rows(X[pruned_gene_set][:, top_n])
    print(f"  Top {n:4d} samples: alpha = {a:.6f}")

In [ ]:
from correlation_vector import cv_eval
import numpy as np

corr_vector, best_idx_local = cv_eval(
    X_part=X[np.array(pruned_gene_set)],
    X_all=X,
    seed=best_samples,
    splits=8
)

ranked_gene_indices = np.argsort(np.abs(corr_vector))[::-1]

print("\nTop 20 genes by |correlation|:")
for i in range(20):
    idx = ranked_gene_indices[i]
    print(f"  Rank {i+1:3d}: gene {idx:5d} | corr = {corr_vector[idx]:+.4f}")
    
    
best_original_gene_ids = np.array(pruned_gene_set)[best_idx_local]

print("\nGenes used to build gene vector (original IDs):")
print(best_original_gene_ids[:20])

In [ ]:
from correlation_vector import (
    cv_eval,
    plot_correlation_vector_distribution,
    plot_correlation_vector_ranked
)

plot_correlation_vector_distribution(
    corr_vector,
    save_path="correlation_vector_distribution.png"
)

plot_correlation_vector_ranked(
    corr_vector,
    save_path="correlation_vector_ranked.png"
)

In [ ]:
from pca import pc1_vec_fun, plot_fork, plot_pc1_summary, pc1_summary_stats

pc1_final = pc1_vec_fun(
    X         = X,
    gene_set  = pruned_gene_set,
    seed_sort = ranked_samples,
    n         = 10
)

pc1_summary_stats(pc1_final)

plot_fork(pc1_final, n_seed=10,
          title="Fork Plot — Python pipeline",
          outlier_pos=9, outlier_label="MGD1_t0_r1")

# full 3-panel summary
plot_pc1_summary(pc1_final, sample_alphas, n_seed=10,
                 outlier_pos=9, outlier_label="MGD1_t0_r1")

In [ ]:
from thresholding import threshold_bic, plot_threshold_summary

bic_genes, bic_samps = threshold_bic(
    cor_vec      = corr_vector,
    sort_order   = ranked_samples,
    pc1          = pc1_final,
    samp_sig     = 0.8,
    sample_names = samples
)

plot_threshold_summary(
    cor_vec      = corr_vector,
    pc1          = pc1_final,
    bic_genes    = bic_genes,
    bic_samps    = bic_samps,
    sort_order   = ranked_samples,
    sample_names = samples
)

In [ ]:
from fork import pc1_align, fork_classifier, print_fork_summary, plot_fork_classified

pc1_aligned = pc1_align(
    X          = X,
    pc1        = pc1_final.copy(),
    cor_vec    = corr_vector,
    sort_order = ranked_samples,
    bic_samps  = bic_samps
)

fork_status = fork_classifier(pc1_aligned, samp_num=len(bic_samps))

print_fork_summary(
    pc1          = pc1_aligned,
    fork_status  = fork_status,
    sort_order   = ranked_samples,
    bic_samps    = bic_samps,
    bic_genes    = bic_genes,
    sample_names = samples
)

plot_fork_classified(
    pc1         = pc1_aligned,
    fork_status = fork_status,
    sort_order  = ranked_samples,
    bic_samps   = bic_samps,
    n_seed      = 10
)

In [ ]:
from multi_run import run_multiple, multi_sample_sort_prep, average_corvec_per_cluster
from silhouette import silhouette_clust_groups, plot_silhouette

# step 1: multiple runs
cor_vec_mat, seeds_list = run_multiple(
    X          = X,
    n_runs     = 20,
    n_genes    = 1000,
    iterations = 500
)

# step 2: silhouette analysis
cluster_groups = silhouette_clust_groups(
    cor_vec_mat  = cor_vec_mat,
    max_clusters = 10,
    plots        = True
)

# step 3: average corvec per cluster
av_corvec = average_corvec_per_cluster(cor_vec_mat, cluster_groups)

# step 4: prepare for SampleSort
top_mats, top_seeds = multi_sample_sort_prep(
    X             = X,
    av_corvec     = av_corvec,
    top_genes_num = 1000,
    groups        = cluster_groups,
    initial_seeds = seeds_list
)

# step 5: SampleSort for each cluster
from sample_sort import extend_bicluster_samples_fast

ranked_per_cluster = []
for k in range(len(cluster_groups)):
    print(f"\nSampleSort for cluster {k+1}...")
    ranked_k, alphas_k, _ = extend_bicluster_samples_fast(
        top_mats[k], np.arange(top_mats[k].shape[0]),
        top_seeds[k], max_rank=None
    )
    ranked_per_cluster.append((ranked_k, alphas_k))

In [ ]:
from sklearn.cluster import KMeans
final_results = []
for k in range(len(cluster_groups)):
    print(f"\n{'='*60}")
    print(f"Cluster {k+1}")
    print(f"{'='*60}")

    ranked_k, alphas_k = ranked_per_cluster[k]
    gem_k          = top_mats[k]
    seed_k         = top_seeds[k]
    gene_indices_k = np.argsort(np.abs(av_corvec[k]))[::-1][:1000]

    # CVEval
    cor_vec_k, _ = cv_eval(
        X_part = gem_k,
        X_all  = X,
        seed   = seed_k,
        splits = 8
    )

    # PC1VecFun
    pc1_k = pc1_vec_fun(
        X         = X,
        gene_set  = gene_indices_k,
        seed_sort = ranked_k,
        n         = 10
    )

    # ThresholdBic
    bic_genes_k, bic_samps_k = threshold_bic(
        cor_vec      = cor_vec_k,
        sort_order   = ranked_k,
        pc1          = pc1_k,
        samp_sig     = 0.8,
        sample_names = samples
    )

    n_bic = len(bic_samps_k) if bic_samps_k is not None else 0

    # PC1Align
    pc1_aligned_k = pc1_align(
        X          = X,
        pc1        = pc1_k.copy(),
        cor_vec    = cor_vec_k,
        sort_order = ranked_k,
        bic_samps  = bic_samps_k
    )

    # ForkClassifier — use pc1_aligned_k not pc1_k
    if n_bic >= 2:
        fork_k = fork_classifier(pc1_aligned_k, samp_num=n_bic)
    else:
        print(f"  Warning: only {n_bic} bicluster samples, skipping")
        fork_k = np.array(["None"] * len(ranked_k), dtype=object)

    # verify fork assignment
    print(f"\nFork verification:")
    print(f"  Upper: {(fork_k == 'Upper').sum()}")
    print(f"  Lower: {(fork_k == 'Lower').sum()}")
    print(f"  None:  {(fork_k == 'None').sum()}")

    # fork plot — pass pc1_aligned_k and fork_k
    plot_fork_classified(
        pc1         = pc1_aligned_k,
        fork_status = fork_k,
        sort_order  = ranked_k,
        bic_samps   = bic_samps_k,
        n_seed      = 10
    )

    final_results.append({
        "cluster"     : k + 1,
        "gene_indices": gene_indices_k,
        "seed"        : seed_k,
        "ranked"      : ranked_k,
        "alphas"      : alphas_k,
        "cor_vec"     : cor_vec_k,
        "pc1"         : pc1_aligned_k.copy(),   # add .copy()
        "bic_genes"   : bic_genes_k.copy(),     # add .copy()
        "bic_samps"   : bic_samps_k.copy() if bic_samps_k is not None else None,
        "fork"        : fork_k.copy()           # add .copy() here
    })

print(f"\n{'='*60}")
print(f"{'Cluster':<10}{'Genes':<10}{'Samples':<10}"
      f"{'Upper':<10}{'Lower':<10}")
print(f"{'='*60}")
for r in final_results:
    n_samps = len(r['bic_samps']) if r['bic_samps'] is not None else 0
    n_upper = int(np.sum(r['fork'] == 'Upper'))
    n_lower = int(np.sum(r['fork'] == 'Lower'))
    print(f"{r['cluster']:<10}"
          f"{r['bic_genes'].sum():<10}"
          f"{n_samps:<10}"
          f"{n_upper:<10}"
          f"{n_lower:<10}")

In [ ]:
from mcbiclust import MCbiclust, MCbiclustMulti
from data_loading import load_ecoli
from preprocessing import remove_samples

# load data
ecoli = load_ecoli("E_coli_v4_Build_6_chips907probes7459.tab")
ecoli = remove_samples(ecoli, ["MGD1_t0_r1"])

# single run
mc = MCbiclust(iterations=1000, random_state=42)
mc.fit(ecoli)
mc.summary()

# multiple runs
mc_multi = MCbiclustMulti(n_runs=20, iterations=500)
mc_multi.fit(ecoli)
mc_multi.summary()

In [ ]:
from cv_plot import cv_plot

top500_loc = np.argsort(np.abs(mc_multi.av_corvec_[0]))[::-1][:500]
n_clusters = len(mc_multi.cluster_groups_)
cnames     = [f"Bicluster {k+1}" for k in range(n_clusters)]

cv_plot(
    cor_vec_list = mc_multi.av_corvec_signed_,  # ← signed version
    geneset_loc  = top500_loc,
    geneset_name = "Top genes",
    alpha1       = 0.01,
    alpha2       = 0.2,
    cnames       = cnames,
    figsize      = (12, 12)
)

In [ ]:
from cv_plot import cv_plot

# top 500 genes from first bicluster as proxy geneset for E. coli
top500_loc = np.argsort(np.abs(av_corvec[0]))[::-1][:500]

cv_plot(
    cor_vec_list = av_corvec,
    geneset_loc  = top500_loc,
    geneset_name = "Top genes",
    alpha1       = 0.01,
    alpha2       = 0.2,
    cnames       = ["Bicluster 1", "Bicluster 2", "Bicluster 3"],
    figsize      = (12, 12)
)

In [ ]:
from point_score import point_score_calc, get_fork_gene_sets, plot_point_score

# get two gene groups from dendrogram split
gene_set1, gene_set2 = get_fork_gene_sets(
    X          = X,
    gene_set   = pruned_gene_set,
    sample_set = best_samples
)

# compute point score
ps_vec = point_score_calc(X, gene_set1, gene_set2)

# plot
plot_point_score(
    ps_vec     = ps_vec,
    pc1        = pc1_aligned,
    sort_order = ranked_samples
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def cv_plot(cor_vec_list, geneset_loc, geneset_name,
            alpha1=0.005, alpha2=0.1, cnames=None,
            figsize=None):
    """
    Python equivalent of CVPlot from MCbiclust.

    cor_vec_list : list of correlation vectors (one per distinct bicluster)
    geneset_loc  : boolean mask or indices of genes in geneset of interest
    geneset_name : name of the geneset (e.g. 'Mitochondrial')
    alpha1       : transparency for non-geneset genes (lower triangle)
    alpha2       : transparency for geneset genes (upper triangle)
    cnames       : optional list of names for each correlation vector
    """
    n_cv = len(cor_vec_list)
    n_genes = len(cor_vec_list[0])

    # column names
    if cnames is None or len(cnames) != n_cv:
        cnames = [f"CV{i+1}" for i in range(n_cv)]

    # geneset mask
    mask = np.zeros(n_genes, dtype=bool)
    mask[geneset_loc] = True
    non_mask = ~mask

    # colours — match R's hue_pal: geneset=coral, non-geneset=steelblue
    col_geneset    = "#F8766D"   # R hue_pal color 1
    col_nongeneset = "#00BFC4"   # R hue_pal color 2

    if figsize is None:
        figsize = (4 * n_cv, 4 * n_cv)

    fig = plt.figure(figsize=figsize)
    gs  = gridspec.GridSpec(n_cv, n_cv, figure=fig,
                            hspace=0.3, wspace=0.3)

    for row in range(n_cv):
        for col in range(n_cv):

            ax = fig.add_subplot(gs[row, col])

            # ── diagonal: density plot ────────────────────────────
            if row == col:
                cv = cor_vec_list[row]
                ax.hist(cv[non_mask], bins=60, density=True,
                        color=col_nongeneset, alpha=0.5,
                        label=f"Non {geneset_name}")
                ax.hist(cv[mask], bins=60, density=True,
                        color=col_geneset, alpha=0.5,
                        label=geneset_name)
                ax.set_xlabel(cnames[row], fontsize=9)
                ax.set_ylabel("Density", fontsize=8)
                if row == 0:
                    ax.legend(fontsize=7, loc="upper left")
                ax.set_title(cnames[row], fontsize=10, fontweight="bold")

            # ── lower triangle: non-geneset genes ─────────────────
            elif row > col:
                x = cor_vec_list[col]
                y = cor_vec_list[row]
                ax.scatter(x[non_mask], y[non_mask],
                           s=2, alpha=alpha1,
                           color=col_nongeneset, rasterized=True)
                ax.set_xlabel(cnames[col], fontsize=8)
                ax.set_ylabel(cnames[row], fontsize=8)
                ax.axhline(0, color="grey", lw=0.5, linestyle="--")
                ax.axvline(0, color="grey", lw=0.5, linestyle="--")

                # correlation annotation
                r = np.corrcoef(x[non_mask], y[non_mask])[0, 1]
                ax.text(0.05, 0.92, f"r={r:.2f}",
                        transform=ax.transAxes, fontsize=8,
                        color=col_nongeneset)

            # ── upper triangle: geneset genes ─────────────────────
            else:
                x = cor_vec_list[col]
                y = cor_vec_list[row]
                ax.scatter(x[mask], y[mask],
                           s=4, alpha=alpha2,
                           color=col_geneset, rasterized=True)
                ax.set_xlabel(cnames[col], fontsize=8)
                ax.set_ylabel(cnames[row], fontsize=8)
                ax.axhline(0, color="grey", lw=0.5, linestyle="--")
                ax.axvline(0, color="grey", lw=0.5, linestyle="--")

                # correlation annotation
                r = np.corrcoef(x[mask], y[mask])[0, 1]
                ax.text(0.05, 0.92, f"r={r:.2f}",
                        transform=ax.transAxes, fontsize=8,
                        color=col_geneset)

            ax.tick_params(labelsize=7)

    fig.suptitle("Correlation Vector Plot", fontsize=14,
                 fontweight="bold", y=1.01)

    plt.savefig("cv_plot.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: cv_plot.png")


# ── run it ────────────────────────────────────────────────────────
# use average correlation vectors from each distinct bicluster
# geneset_loc = indices of genes in your gene set of interest
# for E. coli we don't have a named geneset so use top 500 genes
# from the first bicluster as a proxy

top500_loc = np.argsort(np.abs(av_corvec[0]))[::-1][:500]

cv_plot(
    cor_vec_list = av_corvec,          # 3 average correlation vectors
    geneset_loc  = top500_loc,         # top 500 genes as "geneset"
    geneset_name = "Top genes",
    alpha1       = 0.01,
    alpha2       = 0.2,
    cnames       = ["Bicluster 1", "Bicluster 2", "Bicluster 3"],
    figsize      = (12, 12)
)

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

def go_enrichment_analysis(gene_names, gene_values, gene_sets, sig_rate=0.05):
    """
    Python equivalent of GOEnrichmentAnalysis from MCbiclust.
    Works with any gene set annotations, not just GO terms.

    gene_names  : array of gene names (length = n_genes)
    gene_values : correlation vector values (length = n_genes)
    gene_sets   : dict of {term_name: [list of gene names in term]}
    sig_rate    : adjusted p-value threshold
    """
    gene_names  = np.array(gene_names)
    gene_values = np.array(gene_values, dtype=float)
    av_gene_val = gene_values.mean()

    term_names  = list(gene_sets.keys())
    n_terms     = len(term_names)

    print(f"Testing {n_terms} gene sets...")

    # ── Mann-Whitney test per gene set ────────────────────────────
    # R: MannWhitneyGOTerms
    # for each term: compare gene_values of genes IN term
    # vs genes NOT in term
    pvalues = []
    for term in term_names:
        term_genes  = gene_sets[term]
        in_term     = np.isin(gene_names, term_genes)
        vals_in     = gene_values[in_term]
        vals_out    = gene_values[~in_term]

        if len(vals_in) < 2 or len(vals_out) < 2:
            pvalues.append(1.0)
            continue

        _, p = mannwhitneyu(vals_in, vals_out, alternative="two-sided")
        pvalues.append(float(p))

    pvalues    = np.array(pvalues)

    # ── multiple testing correction (BH by default like R's p.adjust) ──
    _, adj_pvalues, _, _ = multipletests(pvalues, method="fdr_bh")

    # ── filter significant terms ──────────────────────────────────
    sig_mask   = adj_pvalues < sig_rate
    sig_idx    = np.where(sig_mask)[0]
    order      = np.argsort(adj_pvalues[sig_idx])
    sig_idx    = sig_idx[order]

    if len(sig_idx) == 0:
        print("No significant gene sets found.")
        return None

    # ── build results dataframe ───────────────────────────────────
    results = []
    for i in sig_idx:
        term        = term_names[i]
        term_genes  = gene_sets[term]
        in_term     = np.isin(gene_names, term_genes)

        n_in_term   = int(in_term.sum())
        n_term_size = len(term_genes)
        cv_mean     = float(gene_values[in_term].mean()) if n_in_term > 0 else 0.0
        phenotype   = 1 if cv_mean > av_gene_val else -1

        results.append({
            "term":          term,
            "n_in_term":     n_term_size,
            "n_in_genelist": n_in_term,
            "pvalue":        float(pvalues[i]),
            "adj_pvalue":    float(adj_pvalues[i]),
            "cv_mean":       cv_mean,
            "phenotype":     phenotype
        })

    df = pd.DataFrame(results)

    print(f"\nSignificant gene sets: {len(df)} / {n_terms}")
    print(df.to_string(index=False))

    return df


# ── For E. coli: use KEGG pathways or custom gene sets ────────────
# Since we don't have GO annotations for E. coli genes,
# we can test using the initial_gene_set vs rest as a proxy
# or load KEGG pathways if available

# Example with a simple custom gene set test:
gene_names_arr = np.array(genes)   # your gene name array

# create a simple test: initial_gene_set as one "pathway"
gene_sets_test = {
    "Initial_gene_set": [genes[i] for i in initial_gene_set],
    "Bicluster_1_genes": [genes[i] for i in range(len(genes)) 
                          if bic_genes[i]],
}

result_df = go_enrichment_analysis(
    gene_names  = gene_names_arr,
    gene_values = corr_vector,      # from CVEval
    gene_sets   = gene_sets_test,
    sig_rate    = 0.05
)

In [ ]:
import subprocess
subprocess.run(["pip", "install", "goatools", "biopython"], check=True)

In [ ]:
from enrichment import mann_whitney_enrichment, go_enrichment, download_go_files

# ── Mann-Whitney enrichment (no annotation files needed) ──────────
# test if a known gene set is enriched in the correlation vector
sos_genes = ["dinI", "lexA", "recA", "sulA", "umuD",
             "dinP", "lon", "ruvA", "uvrA"]

stat, pval, n_found = mann_whitney_enrichment(
    cor_vec        = final_results[1]["cor_vec"],  # cluster 2 = SOS
    gene_names     = genes,
    gene_set_names = sos_genes
)

# ── full GO enrichment (needs annotation files) ───────────────────
# download files first (only needed once)
download_go_files(obo_path="go-basic.obo",
                  gaf_path="ecocyc.gaf",
                  organism="ecoli")

results_df = go_enrichment(
    cor_vec    = final_results[1]["cor_vec"],
    gene_names = genes,
    obo_file   = "go-basic.obo",
    gaf_file   = "ecocyc.gaf",
    top_n      = 500,
    alpha      = 0.05
)

# E.coli

In [ ]:
## from mcbiclust import MCbiclust, MCbiclustMulti
from data_loading import load_ecoli
from preprocessing import remove_samples


# load data
ecoli = load_ecoli("E_coli_v4_Build_6_chips907probes7459.tab")

# single run
# mc = MCbiclust(iterations=1000, random_state=42)
# mc.fit(ecoli)
# mc.summary()

# multiple runs
mc_multi = MCbiclustMulti(n_runs=1000, iterations=1000)
mc_multi.fit(ecoli)
mc_multi.summary()


In [ ]:
from data_loading import load_ecoli

ecoli = load_ecoli("E_coli_v4_Build_6_chips907probes7459.tab")
samples = ecoli.samples
print(f"Total samples: {len(samples)}")
print("First 20 samples:")
for s in samples[:20]:
    print(f"  {s}")

In [ ]:
import pandas as pd
import numpy as np

def parse_ecoli_metadata(samples):
    metadata = []
    for s in samples:
        parts = s.split("_")
        
        # aeration: U = aerobic, A = anaerobic, K = knockout
        if "_U_" in s:
            aeration = "Aerobic"
        elif "_A_" in s:
            aeration = "Anaerobic"
        elif "_K_" in s:
            aeration = "Knockout"
        else:
            aeration = "Other"
        
        # growth phase from OD
        if "N0025" in s:
            growth_phase = "Early exponential"
        elif "N0075" in s:
            growth_phase = "Mid exponential"
        elif "stat" in s.lower():
            growth_phase = "Stationary"
        elif "biofilm" in s.lower():
            growth_phase = "Biofilm"
        else:
            growth_phase = "Other"
        
        # treatment
        if "norflox" in s.lower() or "norf" in s.lower():
            treatment = "Norfloxacin"
        elif "biofilm" in s.lower():
            treatment = "Biofilm"
        else:
            treatment = "None"
        
        metadata.append({
            "sample":       s,
            "aeration":     aeration,
            "growth_phase": growth_phase,
            "treatment":    treatment
        })
    
    return pd.DataFrame(metadata)

meta = parse_ecoli_metadata(samples)
print(meta["aeration"].value_counts())
print(meta["growth_phase"].value_counts())
print(meta["treatment"].value_counts())

In [ ]:
# see what's in the Other category
other_samples = meta[meta["aeration"] == "Other"]["sample"]
print("Sample of Other aeration samples:")
for s in other_samples[:30]:
    print(f"  {s}")

# also check unique patterns
import re
# extract the middle part of sample names
patterns = set()
for s in samples:
    parts = s.split("_")
    if len(parts) >= 2:
        patterns.add(parts[1])

print("\nAll unique second-part patterns:")
for p in sorted(patterns):
    print(f"  {p}")

In [ ]:
def parse_ecoli_metadata_v2(samples):
    metadata = []
    for s in samples:
        s_lower = s.lower()
        
        # aeration
        if "_U_" in s:
            aeration = "Aerobic"
        elif "_anaerobic" in s_lower or "wtanaerobic" in s_lower or "_kno" in s_lower:
            aeration = "Anaerobic"
        elif "_D_" in s:
            aeration = "Aerobic"  # D = deletion mutant, still aerobic
        else:
            aeration = "Aerobic"  # default most E. coli M3D experiments are aerobic
        
        # growth phase
        if "N0000" in s:
            growth_phase = "Lag"
        elif "N0025" in s:
            growth_phase = "Early exponential"
        elif "N0050" in s:
            growth_phase = "Mid exponential"
        elif "N0075" in s or "N0100" in s:
            growth_phase = "Late exponential"
        elif "N10000" in s or "stat" in s_lower:
            growth_phase = "Stationary"
        elif "biofilm" in s_lower:
            growth_phase = "Biofilm"
        else:
            growth_phase = "Other"
        
        # treatment
        if "norfloxacin" in s_lower or "norf" in s_lower:
            treatment = "Norfloxacin"
        elif "biofilm" in s_lower:
            treatment = "Biofilm"
        elif "ampicillin" in s_lower:
            treatment = "Ampicillin"
        elif "kanamycin" in s_lower:
            treatment = "Kanamycin"
        elif "spectinomycin" in s_lower or "spect" in s_lower:
            treatment = "Spectinomycin"
        elif "cefmec" in s_lower or "cef" in s_lower:
            treatment = "Cefmec"
        else:
            treatment = "None"
        
        metadata.append({
            "sample":       s,
            "aeration":     aeration,
            "growth_phase": growth_phase,
            "treatment":    treatment
        })
    
    return pd.DataFrame(metadata)

meta = parse_ecoli_metadata_v2(samples)
print(meta["aeration"].value_counts())
print(meta["growth_phase"].value_counts())
print(meta["treatment"].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# get bicluster 1 results
bic1 = mc_multi.biclusters_[0]
bic2 = mc_multi.biclusters_[1]

def plot_fork_annotated(bic, meta, title, color_by="growth_phase"):
    pc1 = bic.pc1_
    fork = bic.fork_
    ranked = bic.ranked_  # sample ranking indices
    
    # align metadata to ranked order
    meta_ranked = meta.iloc[ranked].reset_index(drop=True)
    ranks = np.arange(1, len(pc1) + 1)
    
    # colour map
    categories = meta_ranked[color_by].unique()
    cmap = plt.cm.get_cmap("tab10", len(categories))
    color_map = {cat: cmap(i) for i, cat in enumerate(categories)}
    colors = meta_ranked[color_by].map(color_map)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(ranks, pc1, c=colors, s=10, alpha=0.7)
    
    # legend
    for cat, col in color_map.items():
        ax.scatter([], [], c=[col], label=cat, s=30)
    ax.legend(title=color_by, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
    
    ax.set_xlabel("Sample rank")
    ax.set_ylabel("PC1")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(f"fork_{title.replace(' ', '_')}_{color_by}.png", dpi=150)
    plt.show()

# plot bicluster 1 coloured by growth phase
plot_fork_annotated(bic1, meta, "E coli Bicluster 1", color_by="growth_phase")
plot_fork_annotated(bic1, meta, "E coli Bicluster 1", color_by="aeration")
plot_fork_annotated(bic2, meta, "E coli Bicluster 2", color_by="growth_phase")
plot_fork_annotated(bic2, meta, "E coli Bicluster 2", color_by="aeration")

In [ ]:
import pandas as pd

gaf_cols = [
    "db", "db_object_id", "db_object_symbol", "qualifier", "go_id",
    "db_reference", "evidence_code", "with_from", "aspect",
    "db_object_name", "db_object_synonym", "db_object_type",
    "taxon", "date", "assigned_by", "annotation_extension", "gene_product_form_id"
]

gaf = pd.read_csv("ecocyc.gaf", sep="\t", comment="!",
                  header=None, names=gaf_cols, low_memory=False)

print(gaf.shape)
print("\nAspects:")
print(gaf["aspect"].value_counts())
print("\nUnique GO terms:", gaf["go_id"].nunique())
print("Unique genes:", gaf["db_object_symbol"].nunique())
print("\nSample gene symbols:")
print(gaf["db_object_symbol"].unique()[:20])

In [ ]:
# extract gene name from probe ID (part before first underscore)
probe_series = pd.Series(probes)
probe_genes = probe_series.str.split("_").str[0].str.lower()

# match against GAF
gaf_genes_lower = set(gaf["db_object_symbol"].str.lower())
probe_genes_lower = set(probe_genes)

overlap = gaf_genes_lower & probe_genes_lower
print(f"Overlap after extracting gene names: {len(overlap)}")

# create mapping from probe to gene name
probe_to_gene = dict(zip(probes, probe_genes))
print("\nSample mappings:")
for p, g in list(probe_to_gene.items())[:10]:
    print(f"  {p} -> {g}")

In [ ]:
from scipy import stats
import numpy as np

# build probe gene name series
probe_series = pd.Series(probes)
probe_gene_names = probe_series.str.split("_").str[0].str.lower()

# build GO term -> probe indices mapping
gaf["gene_lower"] = gaf["db_object_symbol"].str.lower()

# focus on biological process (P) only
gaf_bp = gaf[gaf["aspect"] == "P"]

go_to_probes = {}
for go_id, group in gaf_bp.groupby("go_id"):
    genes_in_term = set(group["gene_lower"])
    probe_indices = np.where(probe_gene_names.isin(genes_in_term))[0]
    if len(probe_indices) >= 5:  # minimum 5 genes per term
        go_to_probes[go_id] = probe_indices

print(f"GO terms with >= 5 probes: {len(go_to_probes)}")

# run Mann-Whitney on bicluster 1 correlation vector
cor_vec1 = mc_multi.biclusters_[0].cor_vec_
abs_cor_vec1 = np.abs(cor_vec1)

results = []
for go_id, probe_idx in go_to_probes.items():
    in_set = abs_cor_vec1[probe_idx]
    out_set = abs_cor_vec1[np.setdiff1d(np.arange(len(abs_cor_vec1)), probe_idx)]
    stat, pval = stats.mannwhitneyu(in_set, out_set, alternative="greater")
    results.append({"go_id": go_id, "n_genes": len(probe_idx), "pval": pval})

results_df = pd.DataFrame(results)
results_df["pval_adj"] = results_df["pval"] * len(results_df)  # Bonferroni
results_df = results_df.sort_values("pval")
print(results_df.head(20))

In [ ]:
# get GO term names from GAF
go_names = gaf_bp.groupby("go_id")["db_object_name"].first().to_dict()

# add names to results
results_df["go_name"] = results_df["go_id"].map(go_names)

# show top 20 with names
print(results_df[["go_id", "go_name", "n_genes", "pval", "pval_adj"]].head(20).to_string(index=False))

In [ ]:
# download GO term names
import urllib.request
import json

# fetch GO term names via QuickGO API
top_go_ids = results_df["go_id"].head(20).tolist()

go_term_names = {}
for go_id in top_go_ids:
    try:
        url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}"
        req = urllib.request.Request(url, headers={"Accept": "application/json"})
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read())
            go_term_names[go_id] = data["results"][0]["name"]
    except:
        go_term_names[go_id] = "unknown"

results_df["go_term_name"] = results_df["go_id"].map(go_term_names)
print(results_df[["go_id", "go_term_name", "n_genes", "pval", "pval_adj"]].head(20).to_string(index=False))

In [ ]:
from scipy import stats
import numpy as np
import urllib.request
import json

# bicluster 2 correlation vector
cor_vec2 = mc_multi.biclusters_[1].cor_vec_
abs_cor_vec2 = np.abs(cor_vec2)

# Mann-Whitney for bicluster 2
results2 = []
for go_id, probe_idx in go_to_probes.items():
    in_set = abs_cor_vec2[probe_idx]
    out_set = abs_cor_vec2[np.setdiff1d(np.arange(len(abs_cor_vec2)), probe_idx)]
    stat, pval = stats.mannwhitneyu(in_set, out_set, alternative="greater")
    results2.append({"go_id": go_id, "n_genes": len(probe_idx), "pval": pval})

results2_df = pd.DataFrame(results2)
results2_df["pval_adj"] = results2_df["pval"] * len(results2_df)
results2_df = results2_df.sort_values("pval")

# get GO term names
top_go_ids2 = results2_df["go_id"].head(20).tolist()
go_term_names2 = {}
for go_id in top_go_ids2:
    try:
        url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}"
        req = urllib.request.Request(url, headers={"Accept": "application/json"})
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read())
            go_term_names2[go_id] = data["results"][0]["name"]
    except:
        go_term_names2[go_id] = "unknown"

results2_df["go_term_name"] = results2_df["go_id"].map(go_term_names2)
results2_df.to_csv("go_enrichment_bic2.csv", index=False)
print(results2_df[["go_id", "go_term_name", "n_genes", "pval", "pval_adj"]].head(20).to_string(index=False))



results_df.to_csv("go_enrichment_bic1.csv", index=False)

# now get GO names for bicluster 2 top terms
top_go_ids2 = results2_df["go_id"].head(20).tolist()
go_term_names2 = {}
for go_id in top_go_ids2:
    try:
        url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}"
        req = urllib.request.Request(url, headers={"Accept": "application/json"})
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read())
            go_term_names2[go_id] = data["results"][0]["name"]
    except:
        go_term_names2[go_id] = "unknown"

results2_df["go_term_name"] = results2_df["go_id"].map(go_term_names2)
results2_df.to_csv("go_enrichment_bic2.csv", index=False)
print(results2_df[["go_id", "go_term_name", "n_genes", "pval", "pval_adj"]].head(20).to_string(index=False))

# E.coli run on 100 iter

In [ ]:
## from mcbiclust import MCbiclust, MCbiclustMulti
from data_loading import load_ecoli
from preprocessing import remove_samples

# load data
ecoli = load_ecoli("E_coli_v4_Build_6_chips907probes7459.tab")

# single run
# mc = MCbiclust(iterations=1000, random_state=42)
# mc.fit(ecoli)
# mc.summary()

# multiple runs
mc_multi = MCbiclustMulti(n_runs=100, iterations=1000)
mc_multi.fit(ecoli)
mc_multi.summary()


# E.coli single run

In [ ]:
# single run
mc = MCbiclust(iterations=1000, random_state=42)
mc.fit(ecoli)
mc.summary()